In [10]:
import arviz as az
import numpy as np
import pandas as pd
import pymc as pm
import seaborn as sns
import matplotlib.pyplot as plt

print(f"Running on PyMC v{pm.__version__}")

Running on PyMC v5.26.1


In [11]:
%config InlineBackend.figure_format = 'retina'
az.style.use("arviz-darkgrid")
rng = np.random.default_rng(seed=42)

In [12]:
data_dtt_30 = pd.read_csv("Kontrol 30_10.csv", sep = ',')
data_control_30 = pd.read_csv("DTT 30_10.csv", sep = ',')

data_dtt_24 = pd.read_csv("DTT 24_10.csv", sep = ',')
data_control_24 = pd.read_csv("Kontrol 24_10.csv", sep = ',')

data_dtt = pd.concat([data_dtt_30, data_dtt_24], ignore_index = True)
data_control = pd.concat([data_control_30, data_control_24], ignore_index = True)

data_dtt['Площадь сомы'] = data_dtt['Вся площадь']*data_dtt['Площадь сомы (%)']/100
data_control['Площадь сомы'] = data_control['Вся площадь']*data_control['Площадь сомы (%)']/100

In [13]:
data_control['Вся площадь'] = data_control['Вся площадь'] * 0.156**2
data_dtt['Вся площадь'] = data_dtt['Вся площадь'] * 0.156**2

data_control['Площадь сомы'] = data_control['Площадь сомы'] * 0.156**2
data_dtt['Площадь сомы'] = data_dtt['Площадь сомы'] * 0.156**2

data_control['Средняя длина отростков'] = data_control['Средняя длина отростков'] * 0.156
data_dtt['Средняя длина отростков'] = data_dtt['Средняя длина отростков'] * 0.156

In [14]:
data_dtt['group'] = 'DTT'
data_control['group'] = 'Control'
df = pd.concat([data_dtt, data_control]).reset_index()

In [21]:
area_dtt = data_dtt['Средняя длина отростков'].dropna().values
area_control = data_control['Средняя длина отростков'].dropna().values

print(f"DTT group: n={len(area_dtt)}, mean={np.mean(area_dtt):.2f}, std={np.std(area_dtt):.2f}")
print(f"Control group: n={len(area_control)}, mean={np.mean(area_control):.2f}, std={np.std(area_control):.2f}")

DTT group: n=754, mean=9.66, std=3.87
Control group: n=282, mean=9.27, std=4.61


In [ ]:
def count_metric(col):
    area_dtt = data_dtt[col].dropna().values
    area_control = data_control[col].dropna().values

    print(col)
    print(f"DTT group: n={len(area_dtt)}, mean={np.mean(area_dtt):.2f}, std={np.std(area_dtt):.2f}")
    print(f"Control group: n={len(area_control)}, mean={np.mean(area_control):.2f}, std={np.std(area_control):.2f}")

    pooled_mean = np.mean(np.concatenate([area_dtt, area_control]))
    pooled_std = np.std(np.concatenate([area_dtt, area_control]))

    with pm.Model() as model:
        group1_mean = pm.Normal("dtt_mean", mu=pooled_mean, sigma=pooled_std*2)
        group2_mean = pm.Normal("control_mean", mu=pooled_mean, sigma=pooled_std*2)
        
        sigma_low, sigma_high = pooled_std*0.1, pooled_std*5
        group1_std = pm.Uniform("dtt_std", lower=sigma_low, upper=sigma_high)
        group2_std = pm.Uniform("control_std", lower=sigma_low, upper=sigma_high)
        
        nu_minus_one = pm.Exponential("nu_minus_one", 1/29.0)
        nu = pm.Deterministic("nu", nu_minus_one + 1)
        
        lambda_1 = group1_std**-2
        lambda_2 = group2_std**-2
        
        pm.StudentT("dtt_obs", nu=nu, mu=group1_mean, lam=lambda_1, observed=area_dtt)
        pm.StudentT("control_obs", nu=nu, mu=group2_mean, lam=lambda_2, observed=area_control)
        
        diff_means = pm.Deterministic("diff_means (dtt-control)", group1_mean - group2_mean)
        diff_stds = pm.Deterministic("diff_stds", group1_std - group2_std)
        effect_size = pm.Deterministic("effect_size", 
                                      diff_means / np.sqrt((group1_std**2 + group2_std**2)/2))

    with model:
        idata = pm.sample(1000, tune=1000, target_accept=0.9, random_seed=42)

    az.plot_posterior(idata, var_names=["diff_means (dtt-control)", "diff_stds", "effect_size"], ref_val=0)
    plt.savefig(f'posterior_plot_{col}.png', dpi=300, bbox_inches='tight')
    
    summary_df = az.summary(idata, var_names=["diff_means (dtt-control)", "diff_stds", "effect_size"])
    summary_df.to_csv(f'best_summary_{col}.csv')
    print(summary_df)

In [ ]:
col1 = 'Площадь сомы'
count_metric(col1)

In [ ]:
col2 = 'Вся площадь'
count_metric(col2)

In [ ]:
col3 = 'Средняя длина отростков'
count_metric(col3)

In [ ]:
col4 = 'Количество узлов'
count_metric(col4)

In [ ]:
col5 = 'Количество отростков'
count_metric(col5)